# **Introduction to Power BI, Charts, DAX & Creating Reports | Assignment (DA-AG-013)**
### Complete solution in ONE Python notebook

Power BI cannot run inside Python, so the practical questions are solved with **pandas + matplotlib**, producing the same visuals and numbers a Power BI report would. For every practical you also get the **DAX formula** and the **Power BI steps**, in case you want to rebuild it in Power BI Desktop for screenshots.

**How to run (Google Colab)**: upload `Global_Superstore2` (.csv/.xlsx) and `Health_activity_data` (.csv/.xlsx) in the Files panel, set the two file names in the Setup cell, then **Runtime → Run all**. (Q8 uses the small table given in the question, so it needs no file.)

# **THEORY QUESTION**

### **1. Define Power BI and what are the key components of the Power BI ecosystem?**

**Answer**:

**Power BI** is Microsoft's **business intelligence and data visualization platform**. It connects to hundreds of data sources, lets you clean and model the data, build interactive reports and dashboards, and share them securely, helping organisations make data-driven decisions.

| Component | What it is / role |
|---|---|
| **Power BI Desktop** | Free **Windows application** for authoring: connect to data, transform it with **Power Query**, build the **data model** (relationships), write **DAX**, and design report pages. |
| **Power BI Service** | The **cloud (SaaS) platform** (app.powerbi.com) where reports are **published, shared and collaborated on**; also hosts dashboards, apps, workspaces, scheduled refresh, row-level security and alerts. |
| **Power BI Mobile** | Apps for **iOS, Android and Windows** to view and interact with reports and dashboards on the go, with notifications and offline access to recent data. |
| **Power BI Gateway** | A bridge (on-premises data gateway) that lets the cloud service **securely connect to on-premises data sources** (SQL Server, Excel on a network drive, etc.) for **scheduled refresh and live queries** without moving the data to the cloud. |

*Other parts*: Power BI Report Server (on-premises publishing), Report Builder (paginated reports), Power Query, and Power BI Embedded.

### **2. Compare Pie Chart vs Donut Chart and Bar Chart vs Column Chart. When would you prefer one over the other? Give one example for each pair.**

**Answer**:

**Pie vs Donut**

| | Pie chart | Donut chart |
|---|---|---|
| Shape | Full circle divided into slices | Circle with a **hole** in the middle |
| Strength | Simple "part of a whole" view; slightly easier to compare slice angles | Centre can show a **total/KPI label**; can show **several rings**; cleaner look |
| Weakness | Hard to read with many slices | Arc lengths are harder to compare; also poor with many categories |
| Prefer when | 2–5 categories, one measure | You want a headline number in the middle or a modern dashboard look |
| Example | Market share of 3 product categories | Sales % by Region with Total Sales shown in the centre |

*Rule*: use both only for **few categories that add up to 100%**; otherwise use a bar chart.

**Bar vs Column**

| | Bar chart (horizontal) | Column chart (vertical) |
|---|---|---|
| Orientation | Categories on the Y axis, values along X | Categories on the X axis, values along Y |
| Prefer when | **Long category names** or **many categories**; ranking | **Few categories**, **time series** (months, years) or natural left-to-right order |
| Example | Total Sales by Sub-Category (17 names), sorted | Monthly Sales across 12 months |

### **3. Explain the significance of Star schema vs Snowflake schema, Primary key vs Foreign key in relationships (Power BI). Why is cardinality important?**

**Answer**:

**Star schema vs Snowflake schema**

| | Star schema | Snowflake schema |
|---|---|---|
| Structure | One central **fact table** (Sales) joined directly to **denormalised dimension tables** (Product, Customer, Date, Region) | Dimensions are **normalised** into further sub-tables (Product → Sub-Category → Category) |
| Joins | Fewer, simple | More joins |
| Performance in Power BI | **Faster and simpler** DAX, better compression | Slower, more complex |
| Storage / redundancy | Some redundancy in dimensions | Less redundancy |
| Recommendation | **Preferred in Power BI** | Use only when needed; usually flattened into a star in Power Query |

**Primary key vs Foreign key**: a **Primary key** uniquely identifies each row of a table (e.g. `ProductID` in the Product dimension; no duplicates, no blanks). A **Foreign key** is a column in another table that refers to that primary key (e.g. `ProductID` in the Sales fact table, which can repeat). A relationship in Power BI connects the foreign key to the primary key, so filters flow from the dimension to the fact table.

**Why cardinality matters**: cardinality (**one-to-many**, many-to-one, one-to-one, many-to-many) describes how rows match between two tables. The correct setting makes **filters propagate correctly**, prevents **double counting and wrong totals**, lets the engine optimise, and avoids ambiguous paths. One-to-many from dimension to fact is the standard; many-to-many should be avoided or handled with a bridge table.

### **4. Differentiate between Calculated column vs Measure. Also, define Row context and Filter context with simple examples.**

**Answer**:

| | Calculated column | Measure |
|---|---|---|
| Calculated | Row by row when data is **loaded/refreshed** | At **query time**, when the visual is rendered |
| Storage | **Stored in the model** (uses memory) | Not stored; calculated on the fly |
| Context | **Row context** | **Filter context** (from slicers, rows, columns, filters) |
| Reacts to slicers? | No (fixed value per row) | **Yes** (changes with the filters) |
| Use for | Categories/bins, values used on axes, slicers | Aggregations: totals, ratios, percentages, KPIs |
| Example | `Profit Margin = Sales[Profit] / Sales[Sales]` | `Total Profit = SUM(Sales[Profit])` |

**Row context**: DAX evaluates an expression **for one row at a time** and knows the current row. It exists in calculated columns and iterators (`SUMX`). *Example*: in the calculated column `Line Total = Sales[Quantity] * Sales[Price]`, each row uses its own Quantity and Price.

**Filter context**: the **set of filters** applied to the data when a measure is calculated (slicers, report/page/visual filters, row and column headers). *Example*: `Total Sales = SUM(Sales[Sales])` in a table by Region shows each region's sales because the Region row header filters the data; adding a slicer for 2014 further restricts it. `CALCULATE` can modify the filter context, e.g. `CALCULATE([Total Sales], Sales[Region] = "West")`.

### **5. What is the difference between a report and a dashboard in Power BI?**

**Answer**:

| | **Report** | **Dashboard** |
|---|---|---|
| Pages | **One or many pages** | **Single page** (canvas) |
| Source | Built on **one dataset/semantic model** | Can combine visuals (tiles) from **many reports and datasets** |
| Created in | **Power BI Desktop** (or Service) | **Power BI Service only** (not Desktop) |
| Interactivity | Fully interactive: slicers, filters, cross-filtering, drill-through, bookmarks | Limited: tiles are **pinned** visuals; clicking a tile opens the underlying report |
| Purpose | **Detailed analysis and exploration** | **At-a-glance monitoring** of key KPIs |
| Alerts | No | **Data alerts** on tiles, plus Q&A |
| Data | Can be edited/modelled in Desktop | Read-only view of pinned items |

*In short*: a report is a detailed, multi-page, interactive analysis; a dashboard is a one-page summary of the most important pinned visuals.

# **PRACTICAL QUESTIONS**

## **Setup and data loading (Global Superstore)**

In [ ]:
import os, re, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
sns.set_theme(style="whitegrid")
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None
    display = print

# >>> CHANGE THESE TWO FILE NAMES to match the files you uploaded <<<
SUPERSTORE_PATH = "Global_Superstore2.csv"        # .csv or .xlsx
HEALTH_PATH = "Health_activity_data.csv"          # .csv or .xlsx

def read_any(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"'{path}' not found. Upload the file in the Colab Files panel and update the file name above.")
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin1")

def norm(c):
    return re.sub(r"[^a-z0-9]", "", str(c).lower())

def to_num(s):
    if s.dtype.kind in "if":
        return s
    return pd.to_numeric(s.astype(str).str.replace(r"[^0-9.\-]", "", regex=True).replace("", np.nan), errors="coerce")

def money(x, pos=None):
    return f"${x/1e6:.1f}M" if abs(x) >= 1e6 else (f"${x/1e3:.0f}K" if abs(x) >= 1e3 else f"${x:.0f}")

In [ ]:
raw = read_any(SUPERSTORE_PATH)
colmap = {"orderdate": "Order Date", "region": "Region", "category": "Category", "subcategory": "Sub-Category",
          "sales": "Sales", "profit": "Profit", "discount": "Discount", "quantity": "Quantity", "segment": "Segment", "market": "Market"}
ss = raw.rename(columns={c: colmap[norm(c)] for c in raw.columns if norm(c) in colmap}).copy()
need = ["Order Date", "Region", "Sub-Category", "Sales", "Profit", "Discount"]
assert all(c in ss for c in need), f"Missing columns: {[c for c in need if c not in ss]}. Found: {list(raw.columns)}"
for c in ["Sales", "Profit", "Discount"]:
    ss[c] = to_num(ss[c])
ss["Order Date"] = pd.to_datetime(ss["Order Date"], dayfirst=True, errors="coerce")
ss = ss.dropna(subset=["Order Date", "Sales", "Profit"])
ss["Month"] = ss["Order Date"].dt.to_period("M").dt.to_timestamp()
print("Superstore:", ss.shape, "| regions:", ss["Region"].nunique(), "| sub-categories:", ss["Sub-Category"].nunique(),
      "| dates:", ss["Order Date"].min().date(), "to", ss["Order Date"].max().date())
display(ss.head(3))

### **6. Clustered Bar Chart: Total Sales by Sub-Category, and Donut Chart: Sales % by Region.**

**Answer**:

*Power BI steps*: **Clustered bar chart** → Y-axis = `Sub-Category`, X-axis = `Sum of Sales`, sort descending, turn on data labels. **Donut chart** → Legend = `Region`, Values = `Sum of Sales`, Format → Detail labels = *Percent of total*.
*DAX*: `Total Sales = SUM('Global_Superstore2'[Sales])` and `Sales % = DIVIDE([Total Sales], CALCULATE([Total Sales], ALL('Global_Superstore2'[Region])))`

In [ ]:
# ---- Clustered bar chart: Total Sales by Sub-Category ----
sub = ss.groupby("Sub-Category")["Sales"].sum().sort_values()
fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(sub.index, sub.values, color="#4c78a8")
ax.bar_label(bars, labels=[money(v) for v in sub.values], padding=3, fontsize=9)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(money)); ax.set_xlim(0, sub.max() * 1.12)
ax.set_title("Total Sales by Sub-Category", fontsize=14, fontweight="bold"); ax.set_xlabel("Total Sales"); ax.set_ylabel("Sub-Category")
ax.grid(axis="y", visible=False); plt.tight_layout(); plt.show()
print(f"Top: {sub.idxmax()} ({sub.max():,.0f}) | Lowest: {sub.idxmin()} ({sub.min():,.0f})")

# ---- Donut chart: Sales % by Region ----
reg = ss.groupby("Region")["Sales"].sum().sort_values(ascending=False)
pct = reg / reg.sum() * 100
fig, ax = plt.subplots(figsize=(9, 7))
wedges, _ = ax.pie(reg, startangle=90, counterclock=False, wedgeprops=dict(width=0.42, edgecolor="white"), colors=sns.color_palette("tab20", len(reg)))
ax.text(0, 0.07, money(reg.sum()), ha="center", fontsize=20, fontweight="bold"); ax.text(0, -0.1, "Total Sales", ha="center", fontsize=11)
ax.legend(wedges, [f"{r}: {p:.1f}%" for r, p in pct.items()], title="Region (% of sales)", loc="center left", bbox_to_anchor=(1, 0.5))
ax.set_title("Sales % by Region", fontsize=14, fontweight="bold"); plt.tight_layout(); plt.show()
print(f"Largest region: {pct.idxmax()} ({pct.max():.1f}%) | smallest: {pct.idxmin()} ({pct.min():.1f}%)")

### **7. Measures `Total Profit = SUM([Profit])` and `Average Discount = AVERAGE([Discount])`, shown in KPI cards, with a Line Chart of profit trend over months.**

**Answer**:

*DAX measures*
```
Total Profit = SUM('Global_Superstore2'[Profit])
Average Discount = AVERAGE('Global_Superstore2'[Discount])
```
*Power BI steps*: Home → **New measure** (type each formula) → insert two **Card** visuals (Fields = the measure; format Average Discount as Percentage) → insert a **Line chart**: X-axis = `Order Date` (Year → Month hierarchy or Month), Y-axis = `Total Profit`.

In [ ]:
total_profit = ss["Profit"].sum()                  # DAX: SUM([Profit])
avg_discount = ss["Discount"].mean()               # DAX: AVERAGE([Discount])
print(f"Total Profit    = {total_profit:,.2f}")
print(f"Average Discount = {avg_discount:.4f}  ({avg_discount * 100:.2f}%)")

fig, axes = plt.subplots(1, 2, figsize=(9, 2.2))
for ax, (name, val, col) in zip(axes, [("Total Profit", money(total_profit), "#2ca02c"), ("Average Discount", f"{avg_discount * 100:.1f}%", "#ff7f0e")]):
    ax.axis("off"); ax.add_patch(plt.Rectangle((0, 0), 1, 1, color=col, alpha=0.15, transform=ax.transAxes))
    ax.text(0.5, 0.6, val, ha="center", va="center", fontsize=26, fontweight="bold", color=col, transform=ax.transAxes)
    ax.text(0.5, 0.18, name, ha="center", va="center", fontsize=13, transform=ax.transAxes)
plt.show()

monthly_profit = ss.groupby("Month")["Profit"].sum()
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(monthly_profit.index, monthly_profit.values, marker="o", color="#2ca02c", linewidth=2)
ax.axhline(0, color="grey", linewidth=0.8); ax.yaxis.set_major_formatter(mtick.FuncFormatter(money))
ax.set_title("Profit trend over months", fontsize=14, fontweight="bold"); ax.set_xlabel("Month"); ax.set_ylabel("Total Profit")
plt.tight_layout(); plt.show()
print(f"Best month: {monthly_profit.idxmax():%b %Y} ({monthly_profit.max():,.0f}) | weakest month: {monthly_profit.idxmin():%b %Y} ({monthly_profit.min():,.0f})")

### **8. DAX measure for the percentage of total sales by product category** (table given in the question)

**Answer**:

Create the table as `Sales` (or enter it with *Enter Data*), then add the measures:
```
Total Sales = SUM(Sales[Sales_Amount])

% of Total Sales =
DIVIDE(
    SUM(Sales[Sales_Amount]),
    CALCULATE(SUM(Sales[Sales_Amount]), ALL(Sales[Product_category]))
)
```
* `SUM(Sales[Sales_Amount])` is the sales of the **current category** (filter context from the visual's row).
* `CALCULATE(..., ALL(Sales[Product_category]))` **removes the category filter** to get the **grand total**.
* `DIVIDE` divides safely (no error if the total is 0). Format the measure as **Percentage**.
(An alternative is `ALLSELECTED`, which respects slicers.) Use it in a table, bar or donut visual with `Product_category`.

In [ ]:
table8 = pd.DataFrame({
    "Product_category": ["Electronics", "Clothing", "Home Appliances", "Books", "Tables & Chairs", "Toy", "Sports Equipment",
                         "Office Supplies", "Beauty Products", "Garden Supplies", "Jewelry", "Automative"],
    "Sales_Amount": [5000, 3000, 7000, 2000, 8000, 1500, 1200, 1000, 4400, 1000, 1800, 2600]})

grand_total = table8["Sales_Amount"].sum()                                   # CALCULATE(SUM(...), ALL(category))
table8["% of Total Sales"] = (table8["Sales_Amount"] / grand_total * 100).round(2)    # DIVIDE(SUM(...), grand total)
print(table8.sort_values("Sales_Amount", ascending=False).to_string(index=False))
print(f"\nGrand total = {grand_total:,}  |  sum of percentages = {(table8['Sales_Amount'] / grand_total * 100).sum():.2f}%")

t = table8.sort_values("Sales_Amount")
plt.figure(figsize=(9, 5.5))
bars = plt.barh(t["Product_category"], t["% of Total Sales"], color="#4c78a8")
plt.bar_label(bars, labels=[f"{v:.1f}%" for v in t["% of Total Sales"]], padding=3)
plt.xlabel("% of Total Sales"); plt.title("Percentage of total sales by product category", fontsize=13, fontweight="bold")
plt.xlim(0, t["% of Total Sales"].max() * 1.15); plt.tight_layout(); plt.show()

### **9. Total Profit measure, Waterfall chart by Sub-Category, Region slicer, insights and recommendations.**

**Answer**:

*DAX*: `Total Profit = SUM('Global_Superstore2'[Profit])`

*Power BI steps*
1. Home → **New measure** → enter the Total Profit formula above.
2. Insert a **Waterfall chart**: Category = `Sub-Category`, Y-axis = `Total Profit` (sort by Total Profit, descending); set increase = green, decrease = red, total = blue.
3. Insert a **Slicer**: Field = `Region` (style: dropdown or list).
4. Arrange the slicer next to the chart and test by selecting a region; check that the visual updates.

The notebook draws the waterfall and has a Region drop-down that works like the slicer.

In [ ]:
def profit_waterfall(region="All", show=True):
    d = ss if region == "All" else ss[ss["Region"] == region]
    p = d.groupby("Sub-Category")["Profit"].sum().sort_values(ascending=False)
    if show:
        starts = p.cumsum() - p
        fig, ax = plt.subplots(figsize=(13, 5.8))
        ax.bar(p.index, p.values, bottom=starts.values, color=["#2ca02c" if v >= 0 else "#d62728" for v in p.values], width=0.7)
        ax.bar("TOTAL", p.sum(), color="#1f77b4", width=0.7)
        for i, (v, s) in enumerate(zip(p.values, starts.values)):
            ax.text(i, s + v + (p.abs().max() * 0.015 if v >= 0 else -p.abs().max() * 0.015), money(v), ha="center", va="bottom" if v >= 0 else "top", fontsize=8)
        ax.text(len(p), p.sum(), money(p.sum()), ha="center", va="bottom", fontsize=9, fontweight="bold")
        ax.axhline(0, color="black", linewidth=0.8); ax.yaxis.set_major_formatter(mtick.FuncFormatter(money))
        plt.xticks(rotation=60, ha="right"); ax.set_ylabel("Profit")
        ax.set_title(f"Profit contribution by Sub-Category (Waterfall)   |   Region: {region}", fontsize=13, fontweight="bold")
        plt.tight_layout(); plt.show()
    return p

p_all = profit_waterfall("All")
if widgets is not None:
    widgets.interact(profit_waterfall, region=widgets.Dropdown(options=["All"] + sorted(ss["Region"].unique()), description="Region slicer"), show=widgets.fixed(True))

In [ ]:
# ----- business insights and recommendations generated from the data -----
loss = p_all[p_all < 0]
top3 = p_all.head(3)
disc = ss.groupby("Sub-Category")["Discount"].mean()
reg_p = ss.groupby("Region")["Profit"].sum().sort_values()
print("BUSINESS INSIGHTS")
print(f"1. Total profit is {p_all.sum():,.0f}. The top three contributors are {', '.join(f'{k} ({v:,.0f})' for k, v in top3.items())}, "
      f"which together give {top3.sum() / p_all[p_all > 0].sum() * 100:.0f}% of all positive profit.")
if len(loss):
    print(f"2. {len(loss)} sub-categories lose money: {', '.join(f'{k} ({v:,.0f})' for k, v in loss.items())}; "
          f"their average discount is {disc[loss.index].mean() * 100:.1f}% versus {disc.drop(loss.index).mean() * 100:.1f}% for profitable ones.")
else:
    print("2. No sub-category is loss-making overall; the weakest is", p_all.index[-1], f"({p_all.iloc[-1]:,.0f}).")
print(f"3. Region view: weakest region is {reg_p.index[0]} ({reg_p.iloc[0]:,.0f}) and strongest is {reg_p.index[-1]} ({reg_p.iloc[-1]:,.0f}); use the slicer to see which sub-categories drive this.")
print("\nRECOMMENDATIONS")
if len(loss) and disc[loss.index].mean() > disc.drop(loss.index).mean():
    print(f"1. Cap or review discounts on the loss-making sub-categories ({', '.join(loss.index[:3])}): their average discount is higher than for profitable sub-categories.")
elif len(loss):
    print(f"1. Review pricing, product cost and shipping for the loss-making sub-categories ({', '.join(loss.index[:3])}); discounts alone do not explain the losses.")
else:
    print(f"1. Keep discounts under control and watch the weakest sub-category ({p_all.index[-1]}).")
print(f"2. Protect and promote the top profit drivers ({', '.join(top3.index)}) with better stock availability and bundles.")
print(f"3. Investigate costs, pricing and shipping in the weakest region ({reg_p.index[0]}) before spending more on sales there.")

### **10. VitaTrack Wellness: one-page health dashboard (Health_activity_data)**

**Answer**:

The notebook answers the six questions with charts, then combines them in a one-page dashboard. *In Power BI*: Card visuals for KPIs; column/bar charts for lifestyle and risk rates; scatter plot for Sleep vs Steps; matrix with conditional formatting for BMI by Age Group × Gender; clustered column for Heart Rate / BP by Smoking and Alcohol; a calculated column or K-means clusters for segments; slicers for Gender, Age Group and Smoking.

*Useful DAX*:
```
Heart Disease % = DIVIDE(CALCULATE(COUNTROWS(Health), Health[Heart Disease] = "Yes"), COUNTROWS(Health))
Avg BMI = AVERAGE(Health[BMI])
Age Group = SWITCH(TRUE(), Health[Age] < 30, "18-29", Health[Age] < 40, "30-39", Health[Age] < 50, "40-49", Health[Age] < 60, "50-59", "60+")
Balanced Lifestyle = IF(Health[Steps] >= 8000 && Health[Sleep] >= 7 && Health[Sleep] <= 9 && Health[Calories] >= 1600 && Health[Calories] <= 2800, "Balanced", "Not balanced")
```

In [ ]:
h_raw = read_any(HEALTH_PATH)
def pick(*keys):
    for k in keys:
        for c in h_raw.columns:
            if k in norm(c):
                return c
    return None
HC = {"Age": pick("age"), "Gender": pick("gender", "sex"), "BMI": pick("bmi"), "Steps": pick("steps", "stepcount"), "Calories": pick("calor"),
      "Sleep": pick("sleep"), "HeartRate": pick("heartrate", "pulse", "hr"), "BP": pick("bloodpressure", "systolic", "bp"),
      "Smoking": pick("smok"), "Alcohol": pick("alcohol", "drink"), "Exercise": pick("exercise", "physicalactivity", "workout"),
      "Diabetic": pick("diabet"), "HeartDisease": pick("heartdisease", "cardio", "cvd", "heart")}
used = {}
for k, v in HC.items():
    if v is not None and v not in used.values():
        used[k] = v
print("Column mapping:", {k: v for k, v in used.items()})
print("Not found:", [k for k in HC if k not in used])
h = h_raw[list(used.values())].copy(); h.columns = list(used.keys())
hh = lambda *n: all(x in h.columns for x in n)

for c in ["Age", "BMI", "Steps", "Calories", "Sleep", "HeartRate"]:
    if hh(c): h[c] = to_num(h[c])
# blood pressure may be "120/80" text or a single number
if hh("BP"):
    if not pd.api.types.is_numeric_dtype(h["BP"]):
        parts = h["BP"].astype(str).str.extract(r"(\d+)\D+(\d+)")
        if parts.notna().all(axis=1).mean() > 0.5:
            h["Systolic"], h["Diastolic"] = pd.to_numeric(parts[0]), pd.to_numeric(parts[1])
        else:
            h["Systolic"] = to_num(h["BP"])
    else:
        h["Systolic"] = h["BP"]
# yes/no style columns -> readable labels
def yn(s):
    t = s.astype(str).str.strip().str.lower()
    if set(t.unique()) <= {"0", "1", "0.0", "1.0", "nan"}:
        return t.map({"1": "Yes", "1.0": "Yes", "0": "No", "0.0": "No"})
    return s.astype(str).str.strip()
for c in ["Smoking", "Alcohol", "Exercise", "Diabetic", "HeartDisease", "Gender"]:
    if hh(c): h[c] = yn(h[c])
if hh("HeartDisease"):
    h["HD"] = h["HeartDisease"].str.lower().isin(["yes", "y", "true", "1", "present", "positive"]).astype(int)

h = h.dropna(subset=[c for c in ["Age", "BMI"] if hh(c)])
if hh("Age"):
    h["AgeGroup"] = pd.cut(h["Age"], [0, 29, 39, 49, 59, 200], labels=["<30", "30-39", "40-49", "50-59", "60+"])
if hh("BMI"):
    h["BMICategory"] = pd.cut(h["BMI"], [0, 18.5, 25, 30, 100], labels=["Underweight", "Normal", "Overweight", "Obese"], right=False)
print("\nHealth data:", h.shape)
display(h.head())

#### **Q10.1 Are users maintaining a balanced lifestyle (Steps, Sleep, Calories)?**
*Assumed healthy ranges (change them if your teacher gives others)*: Steps ≥ 8,000 per day, Sleep 7–9 hours, Calories 1,600–2,800.

In [ ]:
STEPS_MIN, SLEEP_MIN, SLEEP_MAX, CAL_MIN, CAL_MAX = 8000, 7, 9, 1600, 2800
checks = {}
if hh("Steps"):    checks["Steps >= 8,000"] = h["Steps"] >= STEPS_MIN
if hh("Sleep"):    checks["Sleep 7-9 h"] = h["Sleep"].between(SLEEP_MIN, SLEEP_MAX)
if hh("Calories"): checks["Calories 1,600-2,800"] = h["Calories"].between(CAL_MIN, CAL_MAX)
if checks:
    ok = pd.DataFrame(checks); h["Balanced"] = ok.all(axis=1)
    rates = (ok.mean() * 100).round(1); rates["ALL criteria met"] = h["Balanced"].mean() * 100
    print("% of users meeting each healthy-range criterion:\n", rates.round(1).to_string())
    plt.figure(figsize=(8, 4)); b = plt.bar(rates.index, rates.values, color=["#4c78a8"] * (len(rates) - 1) + ["#2ca02c"])
    plt.bar_label(b, fmt="%.1f%%"); plt.ylim(0, 105); plt.ylabel("% of users"); plt.title("Q1: Lifestyle balance", fontweight="bold"); plt.xticks(rotation=15); plt.tight_layout(); plt.show()
    print(f"INSIGHT: only {h['Balanced'].mean() * 100:.1f}% of users satisfy all three criteria; weakest area: {rates.drop('ALL criteria met').idxmin()} ({rates.drop('ALL criteria met').min():.1f}%).")
else:
    print("Skipped: Steps/Sleep/Calories columns not found.")

#### **Q10.2 Which lifestyle patterns (Smoking, Alcohol, BMI, etc.) indicate heart disease risk?**

In [ ]:
if hh("HD"):
    overall = h["HD"].mean() * 100
    print(f"Overall heart disease rate: {overall:.1f}%")
    factors = [c for c in ["Smoking", "Alcohol", "Exercise", "Diabetic", "BMICategory", "AgeGroup", "Gender"] if c in h.columns or hh(c)]
    risk_rows = []
    fig, axes = plt.subplots(2, 4, figsize=(17, 7)); axes = axes.ravel()
    for ax, f in zip(axes, factors):
        r = h.groupby(f)["HD"].mean() * 100
        ax.bar(r.index.astype(str), r.values, color="#d62728"); ax.axhline(overall, color="grey", linestyle="--")
        ax.set_title(f"Heart disease % by {f}", fontsize=10, fontweight="bold"); ax.tick_params(axis="x", rotation=25)
        for g, v in r.items(): risk_rows.append((f, str(g), v))
    for ax in axes[len(factors):]: ax.axis("off")
    plt.suptitle("Q2: Heart disease rate by lifestyle factor (dashed line = overall)", fontweight="bold"); plt.tight_layout(); plt.show()
    risk = pd.DataFrame(risk_rows, columns=["Factor", "Group", "HeartDisease_%"]).sort_values("HeartDisease_%", ascending=False)
    print("Highest-risk groups:\n", risk.head(6).round(1).to_string(index=False))
    # numeric drivers
    num = [c for c in ["Age", "BMI", "Steps", "Calories", "Sleep", "HeartRate", "Systolic"] if hh(c) or c in h.columns]
    corr = h[num + ["HD"]].corr()["HD"].drop("HD").sort_values(key=abs, ascending=False)
    print("\nCorrelation of numeric measures with heart disease:\n", corr.round(3).to_string())
    print(f"\nINSIGHT: highest observed risk is for '{risk.iloc[0]['Factor']} = {risk.iloc[0]['Group']}' ({risk.iloc[0]['HeartDisease_%']:.1f}% vs {overall:.1f}% overall); "
          f"the numeric measure most associated with heart disease is {corr.index[0]} (r = {corr.iloc[0]:.2f}). Association is not proof of cause.")
else:
    print("Skipped: heart disease column not found.")

#### **Q10.3 Is there a visible relationship between Sleep and Physical Activity?**

In [ ]:
act = "Steps" if hh("Steps") else None
if hh("Sleep") and act:
    r = h[["Sleep", act]].corr().iloc[0, 1]
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
    ax[0].scatter(h["Sleep"], h[act], s=12, alpha=0.4, color="#4c78a8")
    z = np.polyfit(h["Sleep"], h[act], 1); xs = np.linspace(h["Sleep"].min(), h["Sleep"].max(), 50); ax[0].plot(xs, np.polyval(z, xs), color="red")
    ax[0].set(xlabel="Sleep (hours)", ylabel="Steps", title=f"Sleep vs Steps (r = {r:.2f})")
    bins = pd.cut(h["Sleep"], [0, 5, 6, 7, 8, 9, 24]); g = h.groupby(bins)[act].mean()
    ax[1].bar(g.index.astype(str), g.values, color="#2ca02c"); ax[1].set(xlabel="Sleep band (hours)", ylabel="Average steps", title="Average steps by sleep band")
    plt.suptitle("Q3: Sleep vs physical activity", fontweight="bold"); plt.tight_layout(); plt.show()
    strength = "no meaningful" if abs(r) < 0.1 else "a weak" if abs(r) < 0.3 else "a moderate" if abs(r) < 0.5 else "a strong"
    print(f"INSIGHT: correlation between sleep and steps is r = {r:.2f}, i.e. {strength} {'positive' if r > 0 else 'negative'} relationship.")
else:
    print("Skipped: Sleep or Steps column not found.")

#### **Q10.4 How does BMI vary across Age Groups and Genders?**

In [ ]:
if hh("BMI", "Age") and hh("Gender"):
    piv = h.pivot_table(index="AgeGroup", columns="Gender", values="BMI", aggfunc="mean")
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    piv.plot(kind="bar", ax=ax[0]); ax[0].set(ylabel="Average BMI", title="Average BMI by age group and gender"); ax[0].tick_params(axis="x", rotation=0)
    sns.heatmap(piv, annot=True, fmt=".1f", cmap="YlOrRd", ax=ax[1]); ax[1].set_title("BMI heat map")
    plt.suptitle("Q4: BMI across age groups and genders", fontweight="bold"); plt.tight_layout(); plt.show()
    print(piv.round(2).to_string())
    print(f"\nINSIGHT: highest average BMI is in the {piv.stack().idxmax()[0]} age group ({piv.stack().idxmax()[1]}, {piv.stack().max():.1f}); lowest in {piv.stack().idxmin()[0]} ({piv.stack().idxmin()[1]}, {piv.stack().min():.1f}).")
elif hh("BMI", "Age"):
    print(h.groupby("AgeGroup")["BMI"].mean().round(2))
else:
    print("Skipped: BMI/Age/Gender columns not found.")

#### **Q10.5 What is the impact of smoking and alcohol on heart rate and blood pressure?**

In [ ]:
outcomes = [c for c in ["HeartRate", "Systolic", "Diastolic"] if c in h.columns]
drivers = [c for c in ["Smoking", "Alcohol"] if hh(c)]
if outcomes and drivers:
    fig, axes = plt.subplots(len(outcomes), len(drivers), figsize=(5.5 * len(drivers), 3.6 * len(outcomes)), squeeze=False)
    summary = []
    for i, o in enumerate(outcomes):
        for j, dname in enumerate(drivers):
            m = h.groupby(dname)[o].mean(); axes[i, j].bar(m.index.astype(str), m.values, color="#ff7f0e")
            axes[i, j].set_title(f"Avg {o} by {dname}", fontsize=10, fontweight="bold"); axes[i, j].set_ylim(m.min() * 0.9, m.max() * 1.05)
            for g, v in m.items(): summary.append((o, dname, str(g), round(v, 1)))
    plt.suptitle("Q5: Impact of smoking and alcohol", fontweight="bold"); plt.tight_layout(); plt.show()
    s = pd.DataFrame(summary, columns=["Outcome", "Driver", "Group", "Mean"]); print(s.to_string(index=False))
    for o in outcomes:
        for dname in drivers:
            m = h.groupby(dname)[o].mean(); print(f"INSIGHT: average {o} ranges from {m.min():.1f} ({m.idxmin()}) to {m.max():.1f} ({m.idxmax()}) across {dname} groups (difference {m.max() - m.min():.1f}).")
else:
    print("Skipped: need Smoking/Alcohol and Heart Rate/Blood Pressure columns.")

#### **Q10.6 Segment people based on their health activity and suggest lifestyle changes** (K-Means on activity measures)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

feats = [c for c in ["Steps", "Sleep", "Calories", "BMI", "HeartRate"] if c in h.columns]
if len(feats) >= 2:
    Z = StandardScaler().fit_transform(h[feats].fillna(h[feats].median()))
    scores = {k: silhouette_score(Z, KMeans(k, random_state=42, n_init=10).fit_predict(Z)) for k in range(2, 6)}
    best_k = max(scores, key=scores.get)
    print("Silhouette by K:", {k: round(v, 3) for k, v in scores.items()}, "-> chosen K =", best_k)
    h["Segment"] = KMeans(best_k, random_state=42, n_init=10).fit_predict(Z)
    prof = h.groupby("Segment")[feats].mean().round(1); prof["Users"] = h["Segment"].value_counts().sort_index()
    if hh("HD"): prof["HeartDisease_%"] = (h.groupby("Segment")["HD"].mean() * 100).round(1)
    display(prof)

    def advice(row):
        tips = []
        if "Steps" in row and row["Steps"] < STEPS_MIN: tips.append("raise daily steps towards 8,000+")
        if "Sleep" in row and row["Sleep"] < SLEEP_MIN: tips.append("sleep 7-9 hours")
        if "Sleep" in row and row["Sleep"] > SLEEP_MAX: tips.append("avoid oversleeping; keep 7-9 hours")
        if "Calories" in row and row["Calories"] > CAL_MAX: tips.append("reduce calorie intake")
        if "BMI" in row and row["BMI"] >= 25: tips.append("weight management (BMI above 25)")
        if "HeartRate" in row and row["HeartRate"] > 80: tips.append("add cardio training to lower resting heart rate")
        return "; ".join(tips) if tips else "maintain the current healthy routine"
    def label(row):
        z = (row[feats] - prof[feats].mean()) / prof[feats].std(ddof=0).replace(0, 1)
        hi, lo = z.idxmax(), z.idxmin()
        return f"High {hi} / low {lo}" if len(prof) > 1 else "All users"
    prof["Segment name"] = prof.apply(label, axis=1); prof["Suggested lifestyle changes"] = prof.apply(advice, axis=1)
    pd.set_option("display.max_colwidth", 120)
    display(prof[["Segment name", "Users", "Suggested lifestyle changes"]])

    fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
    zprof = (prof[feats] - prof[feats].mean()) / prof[feats].std(ddof=0).replace(0, 1)
    sns.heatmap(zprof, annot=prof[feats], fmt=".0f", cmap="RdYlGn", center=0, ax=ax[0]); ax[0].set_title("Segment profile (colour = relative to other segments)")
    ax[1].pie(prof["Users"], labels=[f"Segment {i}" for i in prof.index], autopct="%1.0f%%", startangle=90); ax[1].set_title("Segment sizes")
    plt.tight_layout(); plt.show()
else:
    print("Skipped: need at least two activity columns.")

#### **One-page Power BI-style dashboard**

In [ ]:
fig = plt.figure(figsize=(18, 11)); g = fig.add_gridspec(3, 12, height_ratios=[0.55, 1.5, 1.5], hspace=0.55, wspace=1.4)
tiles = [("Users", f"{len(h):,}", "#1f77b4")]
if hh("HD"): tiles.append(("Heart disease rate", f"{h['HD'].mean() * 100:.1f}%", "#d62728"))
tiles.append(("Avg BMI", f"{h['BMI'].mean():.1f}", "#ff7f0e"))
if "Balanced" in h: tiles.append(("Balanced lifestyle", f"{h['Balanced'].mean() * 100:.1f}%", "#2ca02c"))
w = 12 // len(tiles)
for i, (n, v, c) in enumerate(tiles):
    ax = fig.add_subplot(g[0, i * w:(i + 1) * w]); ax.axis("off"); ax.add_patch(plt.Rectangle((0.03, 0), 0.94, 1, color=c, alpha=0.15, transform=ax.transAxes))
    ax.text(0.5, 0.58, v, ha="center", va="center", fontsize=22, fontweight="bold", color=c, transform=ax.transAxes); ax.text(0.5, 0.15, n, ha="center", va="center", fontsize=11, transform=ax.transAxes)
slots = [(1, slice(0, 4)), (1, slice(4, 8)), (1, slice(8, 12)), (2, slice(0, 4)), (2, slice(4, 8)), (2, slice(8, 12))]; k = 0
if "Balanced" in h:
    ax = fig.add_subplot(g[slots[k]]); rr = pd.DataFrame(checks).mean() * 100; ax.bar(rr.index, rr.values, color="#4c78a8"); ax.set_title("1. Lifestyle balance (% meeting range)", fontweight="bold", fontsize=10); ax.tick_params(axis="x", rotation=20, labelsize=8); k += 1
if hh("HD") and hh("Smoking"):
    ax = fig.add_subplot(g[slots[k]]); r2 = h.groupby("Smoking")["HD"].mean() * 100; ax.bar(r2.index.astype(str), r2.values, color="#d62728"); ax.set_title("2. Heart disease % by smoking", fontweight="bold", fontsize=10); k += 1
if hh("Sleep") and hh("Steps"):
    ax = fig.add_subplot(g[slots[k]]); ax.scatter(h["Sleep"], h["Steps"], s=6, alpha=0.3); ax.set_title("3. Sleep vs steps", fontweight="bold", fontsize=10); ax.set(xlabel="Sleep", ylabel="Steps"); k += 1
if hh("Gender"):
    ax = fig.add_subplot(g[slots[k]]); sns.heatmap(h.pivot_table(index="AgeGroup", columns="Gender", values="BMI", aggfunc="mean"), annot=True, fmt=".1f", cmap="YlOrRd", ax=ax, cbar=False); ax.set_title("4. BMI by age group and gender", fontweight="bold", fontsize=10); ax.set_ylabel(""); k += 1
if outcomes and hh("Smoking"):
    ax = fig.add_subplot(g[slots[k]]); m2 = h.groupby("Smoking")[outcomes[0]].mean(); ax.bar(m2.index.astype(str), m2.values, color="#ff7f0e"); ax.set_title(f"5. Avg {outcomes[0]} by smoking", fontweight="bold", fontsize=10); ax.set_ylim(m2.min() * 0.9, m2.max() * 1.05); k += 1
if "Segment" in h:
    ax = fig.add_subplot(g[slots[k]]); sz = h["Segment"].value_counts().sort_index(); ax.bar([f"Seg {i}" for i in sz.index], sz.values, color="#9467bd"); ax.set_title("6. Health-activity segments (users)", fontweight="bold", fontsize=10)
fig.suptitle("VitaTrack Wellness | Health & Lifestyle Dashboard", fontsize=17, fontweight="bold")
plt.show()

#### **Summary of actionable insights for VitaTrack Wellness**
Use the INSIGHT lines printed above (they are computed from your data). Typical recommendations: (1) set personalised step and sleep goals for users outside the healthy ranges; (2) target smoking-cessation and alcohol-reduction programmes at the groups with the highest heart-disease rate and blood pressure; (3) send segment-specific lifestyle tips (from the segmentation table) and track the segments over time.